# Notebook 1: LLaMA 2 Fine-Tuning Foundations (Beginner to Intermediate)

## 1. Course Goal
This notebook teaches the full foundation needed before training:
- What LLaMA 2 is
- Why fine-tuning is needed
- Full fine-tuning vs PEFT
- Data design for instruction tuning
- Environment setup and validation

By the end, you will have a clean environment and a validated dataset format.

## 2. What Is LLaMA 2?
### 2.1 Family Overview
LLaMA 2 models are decoder-only transformer LLMs available in multiple sizes (7B, 13B, 70B).

### 2.2 Why It Is Popular
- Strong open ecosystem support
- Works with PEFT, quantization, and transformers
- Can be fine-tuned efficiently for domain-specific tasks

### 2.3 Common Use Cases
- Chat assistants
- Domain QA
- Structured extraction and summarization
- Internal copilots

## 3. Fine-Tuning Concepts
### 3.1 Pretraining vs Fine-Tuning
- Pretraining: broad language knowledge from large corpora
- Fine-tuning: adapt behavior for a specific task/domain

### 3.2 Full Fine-Tuning
Updates all model weights. Very expensive in GPU memory and compute.

### 3.3 Parameter-Efficient Fine-Tuning (PEFT)
Train a small subset of parameters while freezing the base model.

### 3.4 Main Techniques Used in Practice
1. LoRA
2. QLoRA
3. Prefix Tuning
4. Prompt Tuning / P-Tuning v2
5. Adapters
6. IA3
7. DoRA
8. Continued Pretraining

In [ ]:
import platform, os, sys

print("Python version:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Working directory:", os.getcwd())

## 4. Hardware and Memory Planning
### 4.1 Quick Guidance
- LLaMA 2 7B full fine-tuning: generally multi-GPU/high-memory setup
- LLaMA 2 7B QLoRA: possible on a single modern GPU

### 4.2 Precision Choices
- FP32: stable but heavy
- FP16/BF16: faster and lighter
- 8-bit/4-bit quantization: largest memory savings

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("VRAM (GB):", round(props.total_memory/1024**3, 2))
else:
    print("No CUDA GPU detected. You can still run structure and dataset steps on CPU.")

## 5. Environment Setup
### 5.1 Core Libraries
- transformers
- datasets
- peft
- trl
- accelerate
- bitsandbytes

### 5.2 Optional Libraries
- evaluate, scikit-learn for metrics
- wandb or tensorboard for tracking

In [ ]:
# Run this once in a fresh environment
# !pip install -U transformers datasets peft trl accelerate bitsandbytes sentencepiece evaluate scikit-learn

## 6. Dataset Design for Instruction Tuning
### 6.1 High-Quality Data Rules
- Clear instructions
- Correct, concise responses
- Diverse tasks and phrasings
- Remove duplicates and noisy text

### 6.2 Schema Recommendation
Use fields: instruction, input (optional), output

### 6.3 Prompt Template
A robust template helps training consistency and inference stability.

In [ ]:
from datasets import Dataset

raw_samples = [
    {"instruction": "Explain gradient descent in simple terms", "input": "", "output": "Gradient descent updates model parameters step by step to reduce prediction error."},
    {"instruction": "Summarize this text", "input": "Transformers process tokens in parallel and rely on attention.", "output": "Transformers use attention to process token relationships in parallel."},
]

def format_prompt(row):
    if row["input"].strip():
        return f"<s>[INST] {row['instruction']}\nInput: {row['input']} [/INST] {row['output']} </s>"
    return f"<s>[INST] {row['instruction']} [/INST] {row['output']} </s>"

formatted = [{"text": format_prompt(r)} for r in raw_samples]
dataset = Dataset.from_list(formatted)
print(dataset)
print("\nExample formatted sample:")
print(dataset[0]["text"])

## 7. Data Quality Checklist Before Training
1. Instruction is unambiguous
2. Output is factually correct
3. Output style matches target behavior
4. No sensitive data
5. Train/validation split exists
6. Sequence length distribution checked
7. Duplicates removed
8. Unsafe content filtered

In [ ]:
from collections import Counter

lengths = [len(x["text"].split()) for x in dataset]
print("Samples:", len(lengths))
print("Min tokens:", min(lengths), "Max tokens:", max(lengths))
print("Avg tokens:", round(sum(lengths)/len(lengths), 2))

first_words = [x["text"].split()[0] for x in dataset]
print("First token distribution:", Counter(first_words))

## 8. Notes for Beginners
- Start with small subsets to validate pipeline
- Keep prompts consistent between training and inference
- Prefer QLoRA first, full fine-tuning later
- Save tokenizer and config with checkpoints

## 9. Next Step
Proceed to Notebook 2 for full QLoRA training implementation.